# Randomized benchmarking (qubex-emulator)

> This is the qubex-emulator version of the upstream qubex experiment notebook. It uses synthetic `FakeExperiment` results and does not connect to hardware.

This notebook shows a compact randomized benchmarking workflow: prepare a
good single-qubit gate set, run standard RB, and then estimate specific
gate errors with interleaved RB.

## 1. Create an `Experiment`

In [1]:
import numpy as np

from IPython.display import display

import qubex_emulator as qx

exp = qx.Experiment(
    system_id="YOUR_SYSTEM_ID",
    muxes=[0],
    # config_dir="/path/to/qubex-config/config",
    # params_dir="/path/to/qubex-config/params/YOUR_SYSTEM_ID",
)

## 2. Connect to the setup

Connect before you run the prerequisite calibrations.

In [2]:
exp.connect()

FakeExperiment(name='fake-qubex-two-qubit-system', device_id='YOUR_SYSTEM_ID', qubit_labels=('Q00', 'Q01', 'Q02', 'Q03'), qubit_frequencies=(7.157231, 8.032295, 7.812112, 6.944337), qubit_anharmonicities=(-0.393715, -0.487412, -0.421337, -0.365884), readout_frequencies=(6.752, 6.903, 6.844, 6.711), coupling_strength=0.005, qubit_lifetime=(20.0, 20.0), qubit_lifetimes=None, hpi_duration=24.0, pi_duration=24.0, readout_duration=1000.0, rzx90_duration=None, cx_duration=None, single_qubit_fidelity=None, two_qubit_fidelity=None, readout_assignment_error=None, positions=((0.0, 0.0), (1.0, 0.0), (2.0, 0.0), (3.0, 0.0)), calibrated_at=None, metadata={'chip_id': None, 'system_id': 'YOUR_SYSTEM_ID', 'muxes': [0], 'calib_note_path': None, 'calibration_valid_days': None, 'extra_options': {}}, readout_pre_margin=0.0, readout_post_margin=0.0, config_path='', params_path='', property_dir=PosixPath('.'), classifier_dir=PosixPath('.'), classifier_type='gmm', configuration_mode='ge-cr-cr', drag_hpi_puls

## 3. Check the waveform

A waveform check is a quick sanity check before you spend time on benchmarking.

In [3]:
waveform_result = exp.check_waveform()

## 4. Measure a baseline Rabi oscillation

Use the baseline Rabi result to confirm that the drive amplitude is in a usable range.

In [4]:
rabi_result = exp.obtain_rabi_params()

## 5. Calibrate the DRAG half-pi pulse

Run the DRAG half-pi calibration first so the primitive gate set is in good shape.

In [5]:
drag_hpi_result = exp.calibrate_drag_hpi_pulse()

## 6. Repeat the DRAG half-pi pulse

Use a repeated-sequence check to confirm that the calibrated half-pi pulse accumulates as expected.

In [6]:
repeat_drag_hpi = exp.repeat_sequence(exp.drag_hpi_pulse)

## 7. Calibrate the DRAG pi pulse

Next, calibrate the DRAG pi pulse used in the benchmarking gate set.

In [7]:
drag_pi_result = exp.calibrate_drag_pi_pulse()

## 8. Repeat the DRAG pi pulse

Check the calibrated pi pulse with the same repeated-sequence workflow.

In [8]:
repeat_drag_pi = exp.repeat_sequence(exp.drag_pi_pulse)

## 9. Save the calibration note

Save the calibrated pulses before you move on to the benchmarking measurements.

In [9]:
exp.calib_note.save()

## 10. Inspect the calibrated pulses

Plot the DRAG half-pi and pi pulses for the target qubit so you can inspect the cached waveforms directly.

In [10]:
Q0 = exp.qubit_labels[0]

print("target qubit:", Q0)

target qubit: Q00


In [11]:
drag_hpi = exp.drag_hpi_pulse[Q0]
display(drag_hpi.plot())

drag_pi = exp.drag_pi_pulse[Q0]
display(drag_pi.plot())

None

None

## 11. Run standard randomized benchmarking

Use the calibrated single-qubit gate set to estimate the average Clifford error.

In [12]:
rb_result = exp.randomized_benchmarking(
    Q0,
    n_cliffords_range=np.arange(0, 1001, 100),
    n_trials=30,
    save_image=True,
)

## 12. Run interleaved RB for `X90`

Interleave `X90` with the RB sequence to estimate the contribution of that gate.

In [13]:
irb_x90 = exp.interleaved_randomized_benchmarking(
    Q0,
    interleaved_clifford="X90",
    n_cliffords_range=range(0, 1001, 100),
    n_trials=30,
    save_image=True,
)

## 13. Run interleaved RB for `X180`

Repeat the interleaved analysis for `X180`.

In [14]:
irb_x180 = exp.interleaved_randomized_benchmarking(
    Q0,
    interleaved_clifford="X180",
    n_cliffords_range=range(0, 1001, 100),
    n_trials=30,
    save_image=True,
)